# 01. Exploratory Data Analysis
- Big Beautiful Index
- XML bills
- PDF bills

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

_REPO = Path().resolve()
_RESEARCH = _REPO / "docs" / "research" / "financial-semantics"
sys.path.insert(0, str(_RESEARCH))

# bill_title and classify_text used in later cells
from classify_bill import build_financial_df, check_coverage, classify_text  # noqa: E402, F401

from deltatrack.bill_tree import bill_title, normalize_bill  # noqa: E402, F401
from deltatrack.diff_pdf import diff_pdfs  # noqa: E402
from deltatrack.parsers.pdf_anchors import breadcrumb_for, extract_anchors  # noqa: E402
from deltatrack.parsers.pdf_text import extract_clean_pages  # noqa: E402

In [2]:
bbi_df = pd.read_csv("../../../../BillTrax/docs-for-ai/bills.csv")

In [3]:
bbi_df.head()

,id,title,introducedDate,lastActionDate,daysActive,status,policyArea,historySize,summaryLength,actionCount,versionCount,budgetEstimateCount,amendmentCount,relatedBillsCount,committeeCount,sponsorCount
0,119-hr-1,One Big Beautiful Bill Act,2025-05-20,2025-07-04,45.0,Became Public Law No: 119-21.,Economics and Public Finance,1979603,104405,59,6,9,493,29,1,1
1,119-hr-10,Reserved for the Speaker.,2025-01-03,NaN,NaN,NaN,NaN,2514,0,0,0,0,0,0,0,1
2,119-hr-100,Protect the Gig Economy Act of 2025,2025-01-03,2025-01-03,0.0,Referred to the House Committee on the Judiciary.,Law,7175,0,3,1,0,0,0,1,1
3,119-hr-1000,Cyber PIVOTT Act,2025-02-05,2025-09-08,215.0,ASSUMING FIRST SPONSORSHIP - Mrs. Biggs (SC) a...,Government Operations and Politics,17935,0,9,1,1,0,0,2,1
4,119-hr-1001,To provide for a memorandum of understanding t...,2025-02-05,2025-05-14,98.0,Received in the Senate and Read twice and refe...,Water Resources Development,17195,1339,15,4,1,0,1,2,1


In [4]:
print(bbi_df.shape)
bbi_df.columns

(118540, 16)


Index(['id', 'title', 'introducedDate', 'lastActionDate', 'daysActive',
       'status', 'policyArea', 'historySize', 'summaryLength', 'actionCount',
       'versionCount', 'budgetEstimateCount', 'amendmentCount',
       'relatedBillsCount', 'committeeCount', 'sponsorCount'],
      dtype='str')

## BBI Column Reference

The Bill Background Information (BBI) dataset has ~118,540 rows, one per bill.
Each row is derived from a Congress.gov BILLSTATUS XML file (the bulk data record for that bill).

| Column | Description | BILLSTATUS source |
|---|---|---|
| `id` | Bill identifier, e.g. `118-hr-4366` | Constructed from `<congress>`, `<type>`, `<number>` |
| `title` | Official bill title | `<title>` |
| `introducedDate` | Date the bill was introduced | `<introducedDate>` |
| `lastActionDate` | Date of the most recent recorded action | `<latestAction><actionDate>` |
| `daysActive` | Days between `introducedDate` and `lastActionDate` | Calculated |
| `status` | Text description of the most recent action | `<latestAction><text>` |
| `policyArea` | Congress.gov policy area classification | `<policyArea><name>` |
| `historySize` | Byte/character size of the full BILLSTATUS XML file | File size |
| `summaryLength` | Character length of the CRS summary (HTML-formatted) | `len(<summaries><text>)` |
| `actionCount` | Number of recorded legislative actions | Count of `<actions><item>` |
| `versionCount` | Number of available bill text versions | Count of `<textVersions>` entries |
| `budgetEstimateCount` | Number of CBO cost estimates | Count of `<cboCostEstimates>` entries |
| `amendmentCount` | Number of amendments filed | Count of `<amendments>` entries |
| `relatedBillsCount` | Number of related bills identified by Congress.gov | Count of `<relatedBills><item>` |
| `committeeCount` | Number of committee referrals | Count of `<committees><item>` |
| `sponsorCount` | Number of co-sponsors (may exclude primary sponsor) | Count of `<cosponsors>` entries |
| `congress` | Congress number (e.g. `118`) | `<congress>` |
| `chamber` | Originating chamber (`House` or `Senate`) | `<originChamber>` |

**Appropriations filter:** `title.str.contains('Appropriation', case=False)` yields **953 bills**.


In [5]:
bbi_df.isnull().sum()

id                       0
title                    0
introducedDate           0
lastActionDate           5
daysActive               5
status                   5
policyArea             875
historySize              0
summaryLength            0
actionCount              0
versionCount             0
budgetEstimateCount      0
amendmentCount           0
relatedBillsCount        0
committeeCount           0
sponsorCount             0
dtype: int64

In [6]:
# Year coverage
print(bbi_df["introducedDate"].min(), bbi_df["introducedDate"].max())

# Extract congress + bill type
bbi_df[["congress", "bill_type"]] = bbi_df["id"].str.extract(r"^(\d+)-([a-z]+)-")
print(bbi_df["congress"].value_counts().sort_index())
print(bbi_df["bill_type"].value_counts())

2011-01-05 2026-05-22
congress
112    12299
113    10637
114    12063
115    13556
116    16601
117    17828
118    19315
119    16241
Name: count, dtype: int64
bill_type
hr         64879
s          34808
hres        9528
sres        6001
hjres       1142
hconres     1098
sjres        675
sconres      409
Name: count, dtype: int64


- hr — House bills
- s — Senate bills
- hres — House resolutions
- sres — Senate resolutions
- hjres — House joint resolutions
- sjres — Senate joint resolutions
- hconres, sconres — concurrent resolutions

In [7]:
approp_df = bbi_df[bbi_df["title"].str.contains("Appropriation", case=False)]
print(len(approp_df))

953


In [8]:
# Save the list appropriations bills to a CSV file
os.makedirs("data", exist_ok=True)
approp_df.to_csv("data/appropriations_bills.csv", index=False)

## Explore Single Bill / Node Structure (XML parsing)

In [9]:
xml_path = Path("../bills/118-hr-4366/1_reported-in-house.xml")
tree = normalize_bill(xml_path)
df_financial = build_financial_df(tree)
check_coverage(df_financial, tree)
df_financial.head()

FileNotFoundError: [Errno 2] No such file or directory: '..\\bills\\118-hr-4366\\1_reported-in-house.xml'

In [ ]:
tree.nodes[0].__dir__()

In [ ]:
import pandas as pd

# Full tree as a DataFrame — the fastest way to see all fields at once
df_tree = pd.DataFrame(
    [{**vars(n), "display_path": list(n.display_path), "match_path": list(n.match_path)} for n in tree.nodes]
)
df_tree.head(20)

In [ ]:
# What tags exist, and how common are they?
df_tree["tag"].value_counts()

# For appropriations nodes only — compare header_text vs display_path side by side
appro = df_tree[df_tree["tag"].str.startswith("appro")]
appro[["tag", "header_text", "display_path", "division_label", "section_number"]].head(30)

# How deep is display_path for each tag type?
df_tree["path_depth"] = df_tree["display_path"].apply(len)
df_tree.groupby("tag")["path_depth"].value_counts()

## PDF parsing

In [ ]:
pdf_path = Path("../tests/corpus/118-hr-4366/1_reported-in-house.pdf")
pages = extract_clean_pages(pdf_path)

# Page-level overview
print(f"{len(pages)} pages")

# All lines as a flat DataFrame
df_lines = pd.DataFrame(
    [
        {
            "page": page.page_number,
            "line_number": ln.line_number,
            "glyph_size": ln.glyph_size,
            "content_left": ln.geom.content_left if ln.geom else None,
            "text": ln.text,
        }
        for page in pages
        for ln in page.lines
    ]
)
df_lines.head(40)

In [ ]:
anchors = extract_anchors(pages)

df_anchors = pd.DataFrame(
    [
        {
            "kind": a.kind,  # title | section | account | major | agency | grouping | subsection | preamble
            "page": a.page_number,
            "line": a.line_number,
            "division": a.division,
            "text": a.text,
            "breadcrumb": breadcrumb_for(a, anchors),  # equivalent of display_path
        }
        for a in anchors
    ]
)

# What kinds exist and how many?
df_anchors["kind"].value_counts()

In [ ]:
from deltatrack.parsers.pdf_text import extract_clean_pages  # noqa: F401

v1_pages = extract_clean_pages(Path("../tests/corpus/118-hr-4366/1_reported-in-house.pdf"))
v2_pages = extract_clean_pages(Path("../tests/corpus/118-hr-4366/2_engrossed-in-house.pdf"))
pdf_diff = diff_pdfs(v1_pages, v2_pages)

df_hunks = pd.DataFrame(
    [
        {
            "change_type": h.change_type,
            "v1_anchor": h.v1_anchor.text if h.v1_anchor else None,
            "v2_anchor": h.v2_anchor.text if h.v2_anchor else None,
            "v1_breadcrumb": breadcrumb_for(h.v1_anchor, pdf_diff.v1_anchors) if h.v1_anchor else None,
            "v2_breadcrumb": breadcrumb_for(h.v2_anchor, pdf_diff.v2_anchors) if h.v2_anchor else None,
        }
        for h in pdf_diff.hunks
    ]
)
df_hunks

In [ ]:
pdf_diff.hunks[0].__dir__()

In [ ]:
pdf_diff.hunks[0].v1_text

### Compare the text of the first five hunks (pdf) vs first five nodes (xml)

In [ ]:
for h in pdf_diff.hunks[0:5]:
    print(h.v1_text)

In [ ]:
for n in tree.nodes[0:5]:
    print(n.body_text)